# Phân đoạn ảnh bằng U-Net với encoder ResNet18 pretrained
Cải tiến so với `01_unet.ipynb`: thay encoder tự học bằng **ResNet18 pretrained trên ImageNet** (transfer learning), giữ nguyên decoder và skip connection tự viết. Thêm **trọng số lớp `border`** trong loss.

In [1]:
import torch
import common as C
print(C.DEVICE)

cuda


## 1. Dữ liệu
Oxford-IIIT Pet, 3 lớp: `pet`, `background`, `border`. Lần chạy đầu tải dữ liệu (~800 MB) và tạo cache nên mất vài phút.

In [ ]:
IMG_SIZE, BATCH_SIZE = 128, 32
loaders = C.get_loaders(IMG_SIZE, BATCH_SIZE)
print({k: len(v.dataset) for k, v in loaders.items()})
C.show_samples(loaders['val'])

## 2. Mô hình ResNet-U-Net
- **Encoder** = các tầng của ResNet18 pretrained: `conv1` (64 kênh, /2), `layer1` (64, /4), `layer2` (128, /8), `layer3` (256, /16), `layer4` (512, /32). Đã biết nhận ra cạnh, kết cấu, hình dạng từ ImageNet.
- **Decoder** tự viết: `ConvTranspose` nhân đôi kích thước, ghép skip từ tầng encoder cùng độ phân giải, rồi `DoubleConv`. Tầng cuối ConvTranspose đưa về đúng kích thước ảnh vào.
- Đầu ra: Conv 1×1 cho 3 kênh = điểm số 3 lớp tại mỗi pixel.

In [ ]:
model = C.ResNetUNet(n_classes=3, pretrained=True)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M tham số')
print('đầu ra:', tuple(model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE)).shape))

## 3. Huấn luyện
- Loss = CrossEntropy (**trọng số lớp `border` = 3**, thứ tự lớp `[pet, background, border]`) + Dice.
- Learning rate nhỏ hơn bài 01 (1e-3) vì encoder đã pretrained.

In [ ]:
EPOCHS, LR = 20, 1e-3
CLASS_WEIGHTS = [1.0, 1.0, 3.0]
model, hist, t = C.train_model('resnet_unet', model, loaders, EPOCHS, LR, CLASS_WEIGHTS)
print(f'Thời gian train: {t/60:.1f} phút')

In [ ]:
C.plot_history(hist)

## 4. Đánh giá trên tập test (3.669 ảnh)

In [ ]:
res = C.evaluate('resnet_unet', model, loaders['test'], hist, t)

In [ ]:
C.show_predictions(model, loaders['test'], n=6)

## Nhận xét
(Điền sau khi chạy: so sánh mIoU từng lớp với `01_unet.ipynb`, đặc biệt lớp `border`.)